# Inspecionando campos

Às vezes queremos inspecionar os campos de um modelo, ou de uma instância.

Já vimos uma forma de fazer isso para os campos do modelo:

In [1]:
from pydantic import BaseModel

In [2]:
class Circle(BaseModel):
    center_x: int = 0
    center_y: int = 0
    radius: int = 1
    name: str | None = None

In [3]:
print(f"Circle.model_fields: {Circle.model_fields!r}")

Circle.model_fields: {'center_x': FieldInfo(annotation=int, required=False, default=0), 'center_y': FieldInfo(annotation=int, required=False, default=0), 'radius': FieldInfo(annotation=int, required=False, default=1), 'name': FieldInfo(annotation=Union[str, NoneType], required=False, default=None)}


Ao longo do curso, veremos muito mais informação aparecer nos objetos `FieldInfo`.

Também podemos examinar os campos de uma instância.

Claro, podemos fazer isso olhando a representação ou serializando os dados:

In [4]:
c1 = Circle(radius=2)
print(f"c1: {c1!r}")

c1: Circle(center_x=0, center_y=0, radius=2, name=None)


In [5]:
c2 = Circle(name="Unit circle")
print(f"c2: {c2!r}")

c2: Circle(center_x=0, center_y=0, radius=1, name='Unit circle')


In [6]:
print(f"c1.model_dump(): {c1.model_dump()!r}")

c1.model_dump(): {'center_x': 0, 'center_y': 0, 'radius': 2, 'name': None}


In [7]:
print(f"c2.model_dump(): {c2.model_dump()!r}")

c2.model_dump(): {'center_x': 0, 'center_y': 0, 'radius': 1, 'name': 'Unit circle'}


Temos a visão de todos os campos do modelo — alguns preenchidos com dados, outros pelos defaults.

Mas talvez queiramos saber quais campos vieram dos dados e quais vieram dos defaults.

In [8]:
print(f"c1.model_fields_set: {c1.model_fields_set!r}")

c1.model_fields_set: {'radius'}


In [9]:
print(f"c2.model_fields_set: {c2.model_fields_set!r}")

c2.model_fields_set: {'name'}


Recebemos um conjunto com os nomes dos campos que foram preenchidos **a partir dos dados**.

Para saber quais campos vieram dos defaults, não há uma função direta, mas é fácil chegar lá.

Lembre que `model_fields` é um dicionário com todos os nomes de campo como chaves.

Então partimos de todos os nomes e subtraímos os que vieram dos dados.

> 📝 **Nota de revisão (Pydantic 2.11+):** o original usava `c1.model_fields` (pela instância), o que está depreciado.
> Use a classe: `Circle.model_fields` (ou `type(c1).model_fields` quando a classe não é conhecida).

In [10]:
print(f"type(c1).model_fields.keys() - c1.model_fields_set: {type(c1).model_fields.keys() - c1.model_fields_set!r}")

type(c1).model_fields.keys() - c1.model_fields_set: {'name', 'center_y', 'center_x'}


Saber quais campos vieram dos dados pode ser útil em certos casos.

Suponha um endpoint de API que recebe um payload JSON. Alguns itens são opcionais e, do nosso lado, preenchemos os defaults durante
a validação.

Porém, podemos querer devolver o payload do usuário como eco. Se devolvermos o modelo serializado como está, ele incluirá os defaults
que injetamos.

Pode não haver problema nisso, mas talvez exista o requisito de devolver o modelo **sem** os defaults injetados. Nesse caso usamos
`model_fields_set`.

Vejamos um exemplo.

Lembre que, ao serializar um modelo, podemos escolher quais campos incluir ou excluir.

In [11]:
class Model(BaseModel):
    field_1: int = 1
    field_2: int | None = None
    field_3: str
    field_4: str | None = "Python"

In [12]:
m1 = Model(field_3="m1")
m2 = Model(field_1=1, field_2=None, field_3="m2", field_4="Python")
m3 = Model(field_1=10, field_2=20, field_3="m3", field_4="Pydantic")

In [13]:
print(f"m1.model_dump(): {m1.model_dump()!r}")

m1.model_dump(): {'field_1': 1, 'field_2': None, 'field_3': 'm1', 'field_4': 'Python'}


E se quisermos ver só os campos que de fato vieram nos dados desserializados:

In [14]:
print(f"m1.model_dump(include=m1.model_fields_set): {m1.model_dump(include=m1.model_fields_set)!r}")

m1.model_dump(include=m1.model_fields_set): {'field_3': 'm1'}


O mesmo para as outras instâncias:

In [15]:
print(f"m2.model_dump(include=m2.model_fields_set): {m2.model_dump(include=m2.model_fields_set)!r}")

m2.model_dump(include=m2.model_fields_set): {'field_1': 1, 'field_2': None, 'field_3': 'm2', 'field_4': 'Python'}


In [16]:
print(f"m3.model_dump(include=m3.model_fields_set): {m3.model_dump(include=m3.model_fields_set)!r}")

m3.model_dump(include=m3.model_fields_set): {'field_1': 10, 'field_2': 20, 'field_3': 'm3', 'field_4': 'Pydantic'}


> 💡 **Complemento:** o Pydantic tem um atalho para exatamente isso: `model_dump(exclude_unset=True)`. É o que se usa em PATCH de API.

In [17]:
print(f"m1.model_dump(exclude_unset=True) == m1.model_dump(include=m1.model_fields_set): {m1.model_dump(exclude_unset=True) == m1.model_dump(include=m1.model_fields_set)!r}")

m1.model_dump(exclude_unset=True) == m1.model_dump(include=m1.model_fields_set): True
